# PII Sanitization Agent Exploration

Explore validation, local redaction, response parsing, and fail-closed behavior one small cell at a time. No production-module imports are used.

In [1]:
import re

print("re imported")

re imported


In [2]:
REDACTION = "[REDACTED]"
text = "Contact alice@example.com or call 555-123-4567."
print(text)

Contact alice@example.com or call 555-123-4567.


## 1. Validate input

In [3]:
def validate_input(value: str) -> str:
    """Reject empty input before any sanitization call."""

    if not isinstance(value, str) or not value.strip():
        raise ValueError("Text cannot be empty.")
    return value

In [4]:
print(validate_input(text))
try:
    validate_input("   ")
except ValueError as error:
    print(f"Validation passed: {error}")

Contact alice@example.com or call 555-123-4567.
Validation passed: Text cannot be empty.


## 2. Redact common patterns locally

In [5]:
def redact_locally(value: str) -> str:
    """Redact email addresses and phone-like values for offline demos."""

    value = re.sub(r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b", REDACTION, value)
    value = re.sub(r"(?<!\d)(?:\+?\d[\d ()-]{8,}\d)(?!\d)", REDACTION, value)
    return value

In [6]:
sanitized = redact_locally(text)
print(sanitized)
assert "alice@example.com" not in sanitized
assert "555-123-4567" not in sanitized

Contact [REDACTED] or call [REDACTED].


## 3. Fail closed when remote sanitization is unavailable

In [7]:
def fail_closed(error: Exception) -> dict:
    """Return a safe result without exposing the original input."""

    return {
        "status": "failed",
        "sanitized_content": REDACTION,
        "error": str(error),
    }

In [9]:
safe_result = fail_closed(RuntimeError("service unavailable"))
print(safe_result)
assert safe_result["sanitized_content"] == REDACTION

{'status': 'failed', 'sanitized_content': '[REDACTED]', 'error': 'service unavailable'}
